In [ ]:
# Data Engineering Practicals
   #   Practical-7

In [ ]:
# Name: Insiya Shoeb Bobde
# Rollno: 06
# Student-id: 5115304

In [ ]:
# ETL Pipeline from CSV to Database
import pandas as pd
import sqlite3

# -----------------------------
# EXTRACT
# -----------------------------
data = {
    "Name": ["Amit", "Priya", "Rahul", "Sneha", "Karan"],
    "Age": [21, 22, None, 23, 25],
    "City": ["Mumbai", "Pune", "Mumbai", "Delhi", "Pune"],
    "Salary": [30000, 35000, 40000, None, 45000]
}

df = pd.DataFrame(data)
df.to_csv("employees.csv", index=False)

# Read CSV file
df = pd.read_csv("employees.csv")

print("Extracted Data:")
print(df)

# -----------------------------
# TRANSFORM / CLEAN
# -----------------------------

# Fill missing Age with mean
df["Age"] = df["Age"].fillna(df["Age"].mean())

# Fill missing Salary with median
df["Salary"] = df["Salary"].fillna(df["Salary"].median())

# Convert Age to integer
df["Age"] = df["Age"].astype(int)

# Remove duplicate records
df = df.drop_duplicates()

# Create Salary Category
df["Salary_Category"] = df["Salary"].apply(
    lambda x: "High" if x >= 40000 else "Low"
)

print("\nTransformed Data:")
print(df)

# -----------------------------
# LOAD
# -----------------------------

conn = sqlite3.connect("etl_database.db")

df.to_sql(
    "employees",
    conn,
    if_exists="replace",
    index=False
)

print("\nData successfully loaded into database.")

# Verify
result = pd.read_sql("SELECT * FROM employees", conn)
print("\nDatabase Data:")
print(result)

conn.close()

In [ ]:
# Extract Multiple CSV Files and Combine Them
import pandas as pd
import sqlite3

# -----------------------------
# CREATE SAMPLE CSV FILES
# -----------------------------

data1 = {
    "ID": [1, 2, 3],
    "Name": ["Amit", "Priya", "Rahul"],
    "Marks": [85, 90, 78]
}

data2 = {
    "ID": [4, 5, 6],
    "Name": ["Sneha", "Karan", "Neha"],
    "Marks": [88, 76, 95]
}

pd.DataFrame(data1).to_csv("students1.csv", index=False)
pd.DataFrame(data2).to_csv("students2.csv", index=False)

# -----------------------------
# EXTRACT
# -----------------------------

df1 = pd.read_csv("students1.csv")
df2 = pd.read_csv("students2.csv")

print("File 1:")
print(df1)

print("\nFile 2:")
print(df2)

# -----------------------------
# TRANSFORM
# -----------------------------

combined_df = pd.concat([df1, df2], ignore_index=True)

# Create result column
combined_df["Result"] = combined_df["Marks"].apply(
    lambda x: "Pass" if x >= 40 else "Fail"
)

print("\nCombined Dataset:")
print(combined_df)

# -----------------------------
# LOAD
# -----------------------------

conn = sqlite3.connect("student_database.db")

combined_df.to_sql(
    "students",
    conn,
    if_exists="replace",
    index=False
)

print("\nCombined data loaded successfully.")

conn.close()

File 1:
   ID   Name  Marks
0   1   Amit     85
1   2  Priya     90
2   3  Rahul     78

File 2:
   ID   Name  Marks
0   4  Sneha     88
1   5  Karan     76
2   6   Neha     95

Combined Dataset:
   ID   Name  Marks Result
0   1   Amit     85   Pass
1   2  Priya     90   Pass
2   3  Rahul     78   Pass
3   4  Sneha     88   Pass
4   5  Karan     76   Pass
5   6   Neha     95   Pass

Combined data loaded successfully.


In [ ]:
#ETL Pipeline for JSON Data
import json
import pandas as pd
import sqlite3

# -----------------------------
# CREATE SAMPLE JSON DATA
# -----------------------------

json_data = [
    {
        "id": 1,
        "name": "Amit",
        "email": "amit@gmail.com",
        "address": {
            "city": "Mumbai",
            "country": "India"
        }
    },
    {
        "id": 2,
        "name": "Priya",
        "email": "priya@gmail.com",
        "address": {
            "city": "Pune",
            "country": "India"
        }
    }
]

with open("users.json", "w") as file:
    json.dump(json_data, file, indent=4)

# -----------------------------
# EXTRACT
# -----------------------------

with open("users.json", "r") as file:
    data = json.load(file)

print("Extracted JSON:")
print(data)

# -----------------------------
# TRANSFORM
# -----------------------------

transformed_data = []

for user in data:
    transformed_data.append({
        "User_ID": user["id"],
        "Name": user["name"].upper(),
        "Email": user["email"].lower(),
        "City": user["address"]["city"],
        "Country": user["address"]["country"]
    })

df = pd.DataFrame(transformed_data)

print("\nTransformed Data:")
print(df)

# -----------------------------
# LOAD
# -----------------------------

conn = sqlite3.connect("json_database.db")

df.to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

print("\nJSON data loaded into relational database.")

# Verify
result = pd.read_sql("SELECT * FROM users", conn)
print("\nDatabase Table:")
print(result)

conn.close()

Extracted JSON:
[{'id': 1, 'name': 'Amit', 'email': 'amit@gmail.com', 'address': {'city': 'Mumbai', 'country': 'India'}}, {'id': 2, 'name': 'Priya', 'email': 'priya@gmail.com', 'address': {'city': 'Pune', 'country': 'India'}}]

Transformed Data:
   User_ID   Name            Email    City Country
0        1   AMIT   amit@gmail.com  Mumbai   India
1        2  PRIYA  priya@gmail.com    Pune   India

JSON data loaded into relational database.

Database Table:
   User_ID   Name            Email    City Country
0        1   AMIT   amit@gmail.com  Mumbai   India
1        2  PRIYA  priya@gmail.com    Pune   India


In [ ]:
#ETL Process to Identify and Remove Invalid Records
import pandas as pd
import sqlite3

# -----------------------------
# EXTRACT
# -----------------------------

data = {
    "ID": [1, 2, 3, 4, 5],
    "Name": ["Amit", "Priya", "", "Rahul", "Sneha"],
    "Age": [21, -5, 25, 200, 23],
    "Email": [
        "amit@gmail.com",
        "priya@gmail.com",
        "invalidemail",
        "rahul@gmail.com",
        "sneha@gmail.com"
    ]
}

df = pd.DataFrame(data)

print("Original Data:")
print(df)

# -----------------------------
# IDENTIFY INVALID RECORDS
# -----------------------------

invalid = (
    (df["Name"] == "") |
    (df["Age"] < 0) |
    (df["Age"] > 120) |
    (~df["Email"].str.contains("@", na=False))
)

invalid_records = df[invalid]

print("\nInvalid Records:")
print(invalid_records)

# -----------------------------
# REMOVE INVALID RECORDS
# -----------------------------

clean_df = df[~invalid].copy()

print("\nClean Data:")
print(clean_df)

# -----------------------------
# LOAD
# -----------------------------

conn = sqlite3.connect("valid_data.db")

clean_df.to_sql(
    "valid_users",
    conn,
    if_exists="replace",
    index=False
)

print("\nValid records loaded successfully.")

conn.close()

Original Data:
   ID   Name  Age            Email
0   1   Amit   21   amit@gmail.com
1   2  Priya   -5  priya@gmail.com
2   3          25     invalidemail
3   4  Rahul  200  rahul@gmail.com
4   5  Sneha   23  sneha@gmail.com

Invalid Records:
   ID   Name  Age            Email
1   2  Priya   -5  priya@gmail.com
2   3          25     invalidemail
3   4  Rahul  200  rahul@gmail.com

Clean Data:
   ID   Name  Age            Email
0   1   Amit   21   amit@gmail.com
4   5  Sneha   23  sneha@gmail.com

Valid records loaded successfully.


In [ ]:
#Data Validation Before Loading
import pandas as pd
import sqlite3

# -----------------------------
# EXTRACT
# -----------------------------

data = {
    "ID": [1, 2, 3, 4],
    "Name": ["Amit", "Priya", "Rahul", "Sneha"],
    "Age": [21, 22, 150, 24],
    "Email": [
        "amit@gmail.com",
        "priya@gmail.com",
        "rahul@gmail.com",
        "invalid-email"
    ]
}

df = pd.DataFrame(data)

print("Extracted Data:")
print(df)

# -----------------------------
# VALIDATION
# -----------------------------

def validate_record(row):

    if row["Age"] < 18 or row["Age"] > 100:
        return False

    if "@" not in row["Email"]:
        return False

    if row["Name"] == "":
        return False

    return True


df["Valid"] = df.apply(validate_record, axis=1)

print("\nValidation Results:")
print(df)

# Separate valid and invalid records
valid_df = df[df["Valid"] == True].drop(columns=["Valid"])
invalid_df = df[df["Valid"] == False].drop(columns=["Valid"])

print("\nValid Records:")
print(valid_df)

print("\nInvalid Records:")
print(invalid_df)

# -----------------------------
# LOAD ONLY VALID DATA
# -----------------------------

conn = sqlite3.connect("validated_database.db")

valid_df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

print("\nValidated data loaded successfully.")

conn.close()

Extracted Data:
   ID   Name  Age            Email
0   1   Amit   21   amit@gmail.com
1   2  Priya   22  priya@gmail.com
2   3  Rahul  150  rahul@gmail.com
3   4  Sneha   24    invalid-email

Validation Results:
   ID   Name  Age            Email  Valid
0   1   Amit   21   amit@gmail.com   True
1   2  Priya   22  priya@gmail.com   True
2   3  Rahul  150  rahul@gmail.com  False
3   4  Sneha   24    invalid-email  False

Valid Records:
   ID   Name  Age            Email
0   1   Amit   21   amit@gmail.com
1   2  Priya   22  priya@gmail.com

Invalid Records:
   ID   Name  Age            Email
2   3  Rahul  150  rahul@gmail.com
3   4  Sneha   24    invalid-email

Validated data loaded successfully.


In [ ]:
#Incremental Data Loading
import pandas as pd
import sqlite3

# -----------------------------
# INITIAL DATA
# -----------------------------

initial_data = {
    "ID": [1, 2, 3],
    "Name": ["Amit", "Priya", "Rahul"],
    "Salary": [30000, 35000, 40000]
}

df_initial = pd.DataFrame(initial_data)

df_initial.to_csv("employees_initial.csv", index=False)

# -----------------------------
# CREATE DATABASE
# -----------------------------

conn = sqlite3.connect("incremental_database.db")

df_initial.to_sql(
    "employees",
    conn,
    if_exists="replace",
    index=False
)

print("Initial data loaded:")
print(df_initial)

# -----------------------------
# NEW DATA
# -----------------------------

new_data = {
    "ID": [4, 5],
    "Name": ["Sneha", "Karan"],
    "Salary": [42000, 45000]
}

df_new = pd.DataFrame(new_data)

df_new.to_csv("employees_new.csv", index=False)

print("\nNew data:")
print(df_new)

# -----------------------------
# CHECK EXISTING RECORDS
# -----------------------------

existing_df = pd.read_sql(
    "SELECT ID FROM employees",
    conn
)

print("\nExisting IDs:")
print(existing_df)

# -----------------------------
# INCREMENTAL LOAD
# -----------------------------

incremental_df = df_new[
    ~df_new["ID"].isin(existing_df["ID"])
]

if not incremental_df.empty:

    incremental_df.to_sql(
        "employees",
        conn,
        if_exists="append",
        index=False
    )

    print("\nNew records inserted:")
    print(incremental_df)

else:
    print("\nNo new records found.")

# -----------------------------
# FINAL DATABASE
# -----------------------------

final_df = pd.read_sql(
    "SELECT * FROM employees",
    conn
)

print("\nFinal Database:")
print(final_df)

conn.close()

Initial data loaded:
   ID   Name  Salary
0   1   Amit   30000
1   2  Priya   35000
2   3  Rahul   40000

New data:
   ID   Name  Salary
0   4  Sneha   42000
1   5  Karan   45000

Existing IDs:
   ID
0   1
1   2
2   3

New records inserted:
   ID   Name  Salary
0   4  Sneha   42000
1   5  Karan   45000

Final Database:
   ID   Name  Salary
0   1   Amit   30000
1   2  Priya   35000
2   3  Rahul   40000
3   4  Sneha   42000
4   5  Karan   45000
